# 02 — Classification Pipeline Comparison

Compare all classifiers on each patient:
- CSP+LDA (baseline)
- Riemannian: TS+LR, MDM, FgMDM, TS+SVM, TS+LDA
- ACM (Augmented Covariance Method)

5-fold stratified CV, accuracy + Cohen's kappa.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import mne

from src.loading import load_all_patients
from src.classifiers import build_all_pipelines, build_acm_pipeline, evaluate_all
from src.evaluation import compare_to_baseline, full_evaluation
from src.visualization import plot_pipeline_comparison

mne.set_log_level('WARNING')

## 2.1 Load Data

In [ ]:
patient_data = load_all_patients('../data/')

## 2.2 Build Pipelines

In [ ]:
pipelines = build_all_pipelines()

# Add ACM variants
for order in [2, 3, 4]:
    for lag in [5, 7, 9]:
        pipelines[f'ACM(o={order},l={lag})'] = build_acm_pipeline(order=order, lag=lag)

print(f'Total pipelines: {len(pipelines)}')
for name in pipelines:
    print(f'  - {name}')

## 2.3 Per-Patient Evaluation

In [ ]:
all_results = {}

for pid, (X, y, epochs) in patient_data.items():
    print(f'\n{"="*60}')
    print(f'Patient: {pid} — {X.shape[0]} epochs')
    print(f'{"="*60}')
    
    results_df = evaluate_all(X, y, pipelines, n_splits=5)
    results_df = compare_to_baseline(results_df)
    all_results[pid] = results_df
    
    display(results_df[['pipeline', 'mean_acc', 'std_acc', 'vs_baseline_p', 'significant']])

## 2.4 Visualization

In [ ]:
for pid, results_df in all_results.items():
    # Filter to main pipelines for cleaner plot
    main = results_df[~results_df['pipeline'].str.startswith('ACM')].copy()
    plot_pipeline_comparison(main, save_path=f'../results/comparison_{pid}.png')

## 2.5 Best Pipeline — Full Evaluation with Permutation Test

In [ ]:
for pid, (X, y, epochs) in patient_data.items():
    best_name = all_results[pid].iloc[0]['pipeline']
    best_pipe = pipelines[best_name]
    
    print(f'\nBest for {pid}: {best_name}')
    eval_result = full_evaluation(best_pipe, X, y, n_perms=500)

## 2.6 Save Results

In [ ]:
for pid, results_df in all_results.items():
    results_df.to_csv(f'../results/results_{pid}.csv', index=False)
    print(f'Saved results for {pid}')